# Trust Calibration Experiment — Analysis Notebook

This notebook provides a reproducible analysis pipeline for the trust calibration experiment data.
It computes key behavioral trust metrics, runs statistical tests, and generates publication-ready visualizations.

## Metrics Computed
- **Reliance Rate**: Proportion of trials where participant accepted AI recommendation
- **Override Rate**: Proportion of trials where participant overrode AI recommendation
- **Appropriate Reliance**: Accepted when AI was correct
- **Over-trust**: Accepted when AI was incorrect
- **Under-trust**: Overrode when AI was correct
- **Decision Latency**: Response time from scenario display to decision
- **Trust Scale Scores**: Post-task self-report trust ratings

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import json

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 120

print("Libraries loaded successfully.")

## 1. Load Data

In [ ]:
trials = pd.read_csv("sample_data/trials_export.csv")
participants = pd.read_csv("sample_data/participants_export.csv")
trust = pd.read_csv("sample_data/trust_responses_export.csv")

print(f"Trials: {len(trials)} rows")
print(f"Participants: {len(participants)} rows")
print(f"Trust responses: {len(trust)} rows")
print(f"\nConditions: {trials['condition'].unique()}")
print(f"Participants per condition:")
print(participants.groupby('condition').size())

## 2. Reliance Rate by Condition

The primary dependent variable: what proportion of AI recommendations did participants accept?

In [ ]:
# Compute per-participant reliance rate
participant_reliance = trials.groupby(["participant_id", "condition"]).apply(
    lambda g: (g["participant_decision"] == "accept").mean()
).reset_index(name="reliance_rate")

# Summary statistics
reliance_summary = participant_reliance.groupby("condition")["reliance_rate"].agg(
    ["mean", "std", "count"]
).round(3)
reliance_summary.columns = ["Mean Reliance", "SD", "N"]
print("\nReliance Rate by Condition:")
print(reliance_summary)
print()

In [ ]:
# Visualization: Reliance rate by condition
fig, ax = plt.subplots(figsize=(10, 6))

order = ["control", "humanlike_calibrated", "humanlike", "authority"]
labels = ["Control\n(System)", "Humanlike\n(Calibrated)", "Humanlike\n(Overstated)", "Authority\n(Overstated)"]

sns.boxplot(data=participant_reliance, x="condition", y="reliance_rate",
            order=order, ax=ax, width=0.5)
sns.stripplot(data=participant_reliance, x="condition", y="reliance_rate",
              order=order, ax=ax, color=".3", size=4, alpha=0.5, jitter=0.15)

ax.set_xticklabels(labels)
ax.set_ylabel("Reliance Rate (proportion accepted)")
ax.set_xlabel("")
ax.set_title("AI Recommendation Reliance by Experimental Condition")
ax.axhline(y=0.75, color="red", linestyle="--", alpha=0.5, label="AI Accuracy (75%)")
ax.legend()
plt.tight_layout()
plt.savefig("sample_data/reliance_by_condition.png", bbox_inches="tight")
plt.show()

## 3. Override Rate Analysis

In [ ]:
# Compute override rate
participant_override = trials.groupby(["participant_id", "condition"]).apply(
    lambda g: (g["participant_decision"] == "override").mean()
).reset_index(name="override_rate")

override_summary = participant_override.groupby("condition")["override_rate"].agg(
    ["mean", "std"]
).round(3)
override_summary.columns = ["Mean Override Rate", "SD"]
print("Override Rate by Condition:")
print(override_summary)

## 4. Trust Calibration: Appropriate vs. Inappropriate Reliance

Key insight: Do participants appropriately rely on the AI when it's correct and override when it's wrong?

In [ ]:
# Classify each trial into trust calibration categories
def classify_trust(row):
    accepted = row["participant_decision"] == "accept"
    ai_correct = row["ai_is_correct"]
    if accepted and ai_correct:
        return "Appropriate Reliance"
    elif accepted and not ai_correct:
        return "Over-trust"
    elif not accepted and not ai_correct:
        return "Appropriate Override"
    else:
        return "Under-trust"

trials["trust_category"] = trials.apply(classify_trust, axis=1)

# Summary by condition
calibration = trials.groupby(["condition", "trust_category"]).size().unstack(fill_value=0)
calibration_pct = calibration.div(calibration.sum(axis=1), axis=0).round(3) * 100

print("Trust Calibration (% of trials):")
print(calibration_pct)

In [ ]:
# Visualization: stacked bar chart of trust calibration
fig, ax = plt.subplots(figsize=(10, 6))

category_order = ["Appropriate Reliance", "Appropriate Override", "Over-trust", "Under-trust"]
colors = ["#2ecc71", "#3498db", "#e74c3c", "#f39c12"]

calibration_plot = calibration_pct.reindex(columns=category_order).loc[order]
calibration_plot.index = labels
calibration_plot.plot(kind="bar", stacked=True, ax=ax, color=colors, width=0.7)

ax.set_ylabel("Percentage of Trials")
ax.set_xlabel("")
ax.set_title("Trust Calibration Categories by Condition")
ax.legend(title="Category", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.set_xticklabels(labels, rotation=0)
plt.tight_layout()
plt.savefig("sample_data/trust_calibration.png", bbox_inches="tight")
plt.show()

## 5. Decision Latency Analysis

In [ ]:
# Convert to seconds for readability
trials["latency_sec"] = trials["decision_latency_ms"] / 1000

participant_latency = trials.groupby(["participant_id", "condition"])["latency_sec"].mean().reset_index()

latency_summary = participant_latency.groupby("condition")["latency_sec"].agg(
    ["mean", "std"]
).round(2)
latency_summary.columns = ["Mean Latency (s)", "SD"]
print("Mean Decision Latency by Condition:")
print(latency_summary)

In [ ]:
# Visualization: latency by condition and decision type
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# By condition
sns.boxplot(data=participant_latency, x="condition", y="latency_sec",
            order=order, ax=axes[0], width=0.5)
axes[0].set_xticklabels(labels)
axes[0].set_ylabel("Decision Latency (seconds)")
axes[0].set_xlabel("")
axes[0].set_title("Decision Latency by Condition")

# Accept vs Override
sns.boxplot(data=trials, x="condition", y="latency_sec",
            hue="participant_decision", order=order, ax=axes[1], width=0.6)
axes[1].set_xticklabels(labels)
axes[1].set_ylabel("Decision Latency (seconds)")
axes[1].set_xlabel("")
axes[1].set_title("Latency: Accept vs. Override")
axes[1].legend(title="Decision")

plt.tight_layout()
plt.savefig("sample_data/latency_analysis.png", bbox_inches="tight")
plt.show()

## 6. Statistical Tests

In [ ]:
# Chi-square test: is reliance rate independent of condition?
contingency = pd.crosstab(trials["condition"], trials["participant_decision"])
chi2, p_chi, dof, expected = stats.chi2_contingency(contingency)
cramers_v = np.sqrt(chi2 / (len(trials) * (min(contingency.shape) - 1)))

print("=" * 60)
print("Chi-Square Test: Decision ~ Condition")
print(f"  chi2 = {chi2:.2f}, df = {dof}, p = {p_chi:.4f}")
print(f"  Cramer's V = {cramers_v:.3f} (effect size)")
print(f"  {'Significant' if p_chi < 0.05 else 'Not significant'} at alpha = 0.05")
print()

In [ ]:
# One-way ANOVA on reliance rate
groups = [g["reliance_rate"].values for _, g in participant_reliance.groupby("condition")]
f_stat, p_anova = stats.f_oneway(*groups)

# Effect size (eta-squared)
ss_between = sum(len(g) * (g.mean() - participant_reliance["reliance_rate"].mean())**2 for g in groups)
ss_total = sum((participant_reliance["reliance_rate"] - participant_reliance["reliance_rate"].mean())**2)
eta_sq = ss_between / ss_total

print("One-Way ANOVA: Reliance Rate ~ Condition")
print(f"  F({len(groups)-1}, {len(participant_reliance)-len(groups)}) = {f_stat:.2f}, p = {p_anova:.4f}")
print(f"  eta-squared = {eta_sq:.3f} (effect size)")
print(f"  {'Significant' if p_anova < 0.05 else 'Not significant'} at alpha = 0.05")
print()

In [ ]:
# Pairwise comparisons: control vs each treatment
control_reliance = participant_reliance[participant_reliance["condition"] == "control"]["reliance_rate"]

print("Pairwise Comparisons (Control vs. Treatment):")
print("-" * 50)
for cond in ["humanlike", "authority", "humanlike_calibrated"]:
    treatment = participant_reliance[participant_reliance["condition"] == cond]["reliance_rate"]
    t_stat, p_val = stats.ttest_ind(control_reliance, treatment)
    cohens_d = (treatment.mean() - control_reliance.mean()) / np.sqrt(
        (control_reliance.std()**2 + treatment.std()**2) / 2
    )
    print(f"  Control vs {cond}:")
    print(f"    t = {t_stat:.2f}, p = {p_val:.4f}, Cohen's d = {cohens_d:.3f}")
    print()

## 7. Trust Survey Analysis

Post-task self-report trust scale (7 items, item 7 reverse-coded).

In [ ]:
# Reverse-code item 7 (index 6)
trust_scored = trust.copy()
trust_scored.loc[trust_scored["item_index"] == 6, "response"] = \
    6 - trust_scored.loc[trust_scored["item_index"] == 6, "response"]

# Compute mean trust score per participant
participant_trust = trust_scored.groupby(["participant_id", "condition"])["response"].mean().reset_index()
participant_trust.columns = ["participant_id", "condition", "trust_score"]

trust_by_condition = participant_trust.groupby("condition")["trust_score"].agg(
    ["mean", "std"]
).round(3)
trust_by_condition.columns = ["Mean Trust Score", "SD"]
print("Trust Scale Scores by Condition:")
print(trust_by_condition)

In [ ]:
# Visualization: trust scores
fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(data=participant_trust, x="condition", y="trust_score",
            order=order, ax=ax, width=0.5)
sns.stripplot(data=participant_trust, x="condition", y="trust_score",
              order=order, ax=ax, color=".3", size=4, alpha=0.5, jitter=0.15)

ax.set_xticklabels(labels)
ax.set_ylabel("Mean Trust Score (1-5)")
ax.set_xlabel("")
ax.set_title("Self-Reported Trust by Condition")
ax.set_ylim(0.5, 5.5)
plt.tight_layout()
plt.savefig("sample_data/trust_scores.png", bbox_inches="tight")
plt.show()

## 8. Trust Scale vs. Behavioral Reliance

Does self-reported trust predict actual behavioral reliance?

In [ ]:
# Merge trust scores with reliance rates
merged = participant_reliance.merge(participant_trust, on=["participant_id", "condition"])

# Pearson correlation
r, p_corr = stats.pearsonr(merged["trust_score"], merged["reliance_rate"])

fig, ax = plt.subplots(figsize=(8, 6))
for cond in order:
    subset = merged[merged["condition"] == cond]
    ax.scatter(subset["trust_score"], subset["reliance_rate"],
               label=cond, alpha=0.6, s=40)

# Regression line
z = np.polyfit(merged["trust_score"], merged["reliance_rate"], 1)
p = np.poly1d(z)
x_line = np.linspace(merged["trust_score"].min(), merged["trust_score"].max(), 100)
ax.plot(x_line, p(x_line), "--", color="gray", alpha=0.7)

ax.set_xlabel("Self-Reported Trust Score")
ax.set_ylabel("Behavioral Reliance Rate")
ax.set_title(f"Trust Scale vs. Behavioral Reliance (r = {r:.3f}, p = {p_corr:.4f})")
ax.legend(title="Condition")
plt.tight_layout()
plt.savefig("sample_data/trust_vs_reliance.png", bbox_inches="tight")
plt.show()

## 9. Summary Table

In [ ]:
# Build comprehensive summary table
summary = participant_reliance.groupby("condition").agg(
    N=("participant_id", "count"),
    Reliance_Mean=("reliance_rate", "mean"),
    Reliance_SD=("reliance_rate", "std"),
).round(3)

latency_agg = participant_latency.groupby("condition")["latency_sec"].mean().round(2)
trust_agg = participant_trust.groupby("condition")["trust_score"].mean().round(2)

summary["Latency_Mean_s"] = latency_agg
summary["Trust_Mean"] = trust_agg

# Over-trust rate per condition
overtrust_rate = trials[trials["trust_category"] == "Over-trust"].groupby(
    "condition"
).size() / trials.groupby("condition").size()
summary["Overtrust_Rate"] = overtrust_rate.round(3)

print("\n" + "=" * 70)
print("COMPREHENSIVE SUMMARY")
print("=" * 70)
print(summary.to_string())
print("\n")